# Krylov Subspace Methods

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/linear_systems/iterative_methods/krylov_subspace_methods.ipynb)

In [20]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## The Subspace Philosophy

Unlike stationary methods (which continually update a single guess vector), **Krylov subspace methods** construct a growing set of vector *bases* (directions) and search within that spanning subspace for the optimal solution. 

Because we are building a basis for an $n$-dimensional vector space, we would theoretically need at most $n$ basis vectors. However, the true power of these methods is that they often converge to an acceptable tolerance using significantly *fewer* than $n$ vectors!

## Conjugate Gradient (CG)

The Conjugate Gradient (CG) method is the gold standard for solving **symmetric positive definite (SPD)** matrices.

Instead of directly solving $\mathbf{A}\mathbf{x}=\mathbf{b}$, we can reframe the problem as an optimization task. Consider the quadratic surface defined by:

$$ f(\mathbf{x}^{(k)}) = \frac{1}{2} {\mathbf{x}^{(k)}}^T \mathbf{A} \mathbf{x}^{(k)} - \mathbf{b}^T \mathbf{x}^{(k)} $$

This surface has an extremum where its gradient is zero:

$$
\begin{aligned}
\nabla f = \frac{d f}{ d \mathbf{x}} = \vec{0} &= \mathbf{A} \mathbf{x} - \mathbf{b} \\
\vec{0} &= \mathbf{r}
\end{aligned}
$$

Notice that the gradient is exactly the **residual** $\mathbf{r}$. Therefore, finding the minimum of the surface $f$ is mathematically identical to solving the linear system!

### The surface conception

Why can we now think of a quadratic surface? Why does the matrix $\mathbf{A}$ need to be symmetric positive definite? This geometric interpretation is only valid if we can think of $\mathbf{A}$ as the matrix of mixed second derivatives of the surface $f$, $\begin{bmatrix}
\frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2}  \\
\frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} \\
\end{bmatrix}$ (called the Hessian of $f$). This is only possible if $\mathbf{A}$ is symmetric!

Furthermore, if $\mathbf{A}$ is **positive definite** ($\mathbf{x}^T \mathbf{A} \mathbf{x} > 0$), the quadratic surface is strictly convex (opening upwards like a bowl). This guarantees that the extremum is a global *minimum*, and that walking "downhill" on the surface will inevitably lead us to the exact solution!

> If $\mathbf{A}$ isn't positive definite the surface is called a *saddle point* because if curves upwards in some direction and downwards in others. 

### The Search Algorithm

To walk down the bowl towards the minimum, we start at a guess $\mathbf{x}^{(k)}$. We must choose a **step direction** $\mathbf{s}^{(k)}$ and a **step length** $\alpha_k$ to reach our next, better guess $\mathbf{x}^{(k+1)}$:

$$ \mathbf{x}^{(k+1)} = \mathbf{x}^{(k)} + \alpha_k \mathbf{s}^{(k)} $$

The magic of these methods lies entirely in how we choose the step direction. To understand this, we will approach this problem backwards: 

1. Find the optimal $\alpha_k$ assuming we already have $\mathbf{s}^{(k)}$
2. Find the suitable direction $\mathbf{s}^{(k)}$

#### Choosing the Step Length

Given a step direction, an obvious choice for the length $\alpha_k$ is whatever minimizes $f(\mathbf{x}^{(k+1)})$ along that specific ray. By setting the derivative with respect to $\alpha_k$ to zero, we find the optimal step length:

$$
\begin{align}
f(\vec{x}+\alpha^k \vec{s}^k) &= \frac{1}{2} [\vec{x}+\alpha^k \vec{s}^k]^T A [\vec{x}+\alpha^k \vec{s}^k] -\vec{b}^T [\vec{x}+\alpha^k \vec{s}^k] \\
\frac{\partial f}{\partial \alpha^k} = 0 &= {\vec{s}^k}^T A [\vec{x}+\alpha^k \vec{s}^k] -\vec{b}^T \vec{s}^k\\
&= {\vec{s}^k}^T A \vec{x}^k+ {\vec{s}^k}^T A \alpha^k \vec{s}^k -{\vec{s}^k}^T \vec{b} \\
&= {\vec{s}^k}^T [A \vec{x}^k-\vec{b}] + \alpha^k {\vec{s}^k}^T A  \vec{s}^k 
\end{align}
$$

and therefore the optimal step size is:
$$
\begin{aligned}
\alpha_k &= \frac{{\mathbf{s}^{(k)}}^T \mathbf{r}^{(k)}}{{\mathbf{s}^{(k)}}^T \mathbf{A}  \mathbf{s}^{(k)}}
\end{aligned}
$$

So, given *any* step direction $\mathbf{s}^{(k)}$, we can always calculate the mathematically optimal distance to travel.

#### Step Direction: Steepest Descent

The most intuitive choice is to simply walk straight downhill following the steepest gradient of the surface. We already proved the gradient is the residual, so:

$$ \mathbf{s}^{(k)} = -\nabla f = \mathbf{r}^{(k)} $$

Substituting this into our step length formula yields the classic **Steepest Descent** algorithm! Let's code it up and visualize the path it takes down the 2D contour.

### The Algorithm

If we code this mathematical intuition directly into Python, we get the Steepest Descent algorithm.

Let's test it on a sample quadratic surface. Notice the *zig-zag* path the Steepest Descent algorithm takes. It frantically zig-zags back and forth across the ravine! Why? Because taking the steepest path down often forces the next step to partially *undo* the progress of the previous step. In an $n$-dimensional space, this inefficiency becomes catastrophic.

In [21]:
A = np.array([[4, 1], [1, 3]])
b = np.array([5, 6])
x0 = np.array([0., 0.])

# Calculate Steepest Descent
x_final, iterations = exe.steepest_descent(A, b, x0, track_history=True)

# Visualize the path over the purely quadratic surface
fig = exe.visualize_convergence_2d(A, b, iterations, surface_type='quadratic')
fig.show()

Converged after 11 iterations.


#### Step Direction: Conjugate Gradient

To fix the zig-zagging, we must choose step directions that strictly **do not undo each other**. We enforce this by requiring that every new step direction is *conjugate* (orthogonal with respect to $\mathbf{A}$) to all previous step directions:

$$ {\mathbf{s}^{(k+1)}}^T \mathbf{A} \mathbf{s}^{(k)} = 0 $$

This ensures the steps are linearly independent. By building the next direction from a combination of the current residual and the previous direction, we derive the **Conjugate Gradient** update rule:

$$ \mathbf{s}^{(k+1)} = \mathbf{r}^{(k+1)} - \frac{{\mathbf{r}^{(k+1)}}^T \mathbf{A} \mathbf{s}^{(k)}}{{\mathbf{s}^{(k)}}^T \mathbf{A} \mathbf{s}^{(k)}} \mathbf{s}^{(k)} $$

Let's use `scipy.sparse.linalg.cg` to see how beautifully this solves the zig-zag problem!

In [22]:
from scipy.sparse.linalg import cg

A = np.array([[4, 1], [1, 3]])
b = np.array([5, 6])
x0 = np.array([0., 0.])

# We use a clean tracker from our package to avoid boilerplate callbacks
tracker = exe.IterationTracker(x0)

# Execute the native SciPy solver
solution, info = cg(A, b, x0=x0, callback=tracker)

# Visualize the path over the purely quadratic surface
fig = exe.visualize_convergence_2d(A, b, tracker.iterations, surface_type='quadratic')
fig.show()

iterations = np.array(tracker.iterations)
print('Converged in ', len(tracker.iterations)-1, ' steps.')

Converged in  2  steps.


### Why CG is so Powerful

As shown in the plot, CG walks directly to the center of the bowl. We can mathematically verify that the steps are indeed conjugate in $\mathbf{A}$, and that the residuals are orthogonal:

In [23]:
s1 = iterations[1]-iterations[0]
s2 = iterations[2]-iterations[1]
print(s1,s2)
print(np.dot(s1,s2))  #not zero
print(np.dot(s1,A@s2))  #Zero
print(np.dot(s2,A@s1))  #Zero

[1.1381 1.3657] [-0.3199  0.3616]
0.1297882196885316
6.661338147750939e-16
6.661338147750939e-16


In [24]:
r0 = A@iterations[0]-b
r1 = A@iterations[1]-b
r2 = A@iterations[2]-b

print(r0.dot(r1))
print(r1.dot(r2))

0.0
0.0


Because the Conjugate Gradient method enforces strict linear independence of its bases, it is mathematically guaranteed to find the exact solution (barring round-off error) in at most $n$ iterations! 

However, its true strength is that for large sparse matrices, it usually converges to a tight tolerance in far fewer than $n$ iterations.

> **Matrix-Free Algorithms:** Notice that CG only ever uses $\mathbf{A}$ to compute matrix-vector products ($\mathbf{A} \mathbf{x}$). If we can calculate $\mathbf{A}\mathbf{x}$ directly on-the-fly without ever assembling or storing the massive dense matrix $\mathbf{A}$, we can save colossal amounts of RAM. This is the foundation of matrix-free HPC solvers!

### Is CG a direct method?

Note the *space* of the solution vector $x$ is just its dimension $n$, and if CG determins linearly independent vectors, it can only find, at most $n$! Therefore, CG will find the exact solution in $n$ iterations (baring roundoff error) which some may take to imply it is a *direct method*. However, the situation is even better, which qualifies it as an iterative technique! Let's see what happens with a larger system:

In [25]:
import numpy as np
from scipy.sparse.linalg import cg

n = 8
A_r = np.random.rand(n, n)
A_rand = A_r @ A_r.T  # Ensure A is symmetric positive definite
b_rand = np.random.rand(n)
x0_rand = np.zeros(n)


class it_counter(object):
    def __init__(self, disp=True):
        self._disp = disp
        self.niter = 0
    def __call__(self, rk=None):
        self.niter += 1
        if self._disp:
            print('iter %3i\trk = %s' % (self.niter, str(rk)))

np.set_printoptions(precision=4)

solution, info = cg(A_rand, b_rand, x0=x0_rand, atol = 1e-6, rtol = 1e-10, callback = it_counter())
print("Solution:", solution)
print("Info:", info)

iter   1	rk = [0.0257 0.0049 0.0241 0.0252 0.0566 0.0495 0.0689 0.0367]
iter   2	rk = [-0.0851 -0.5817 -0.2945 -0.1461  0.155   0.466   0.6693 -0.0062]
iter   3	rk = [ 0.3513 -2.9009 -1.0088 -1.9964  0.3116  2.091   1.5459  1.4956]
iter   4	rk = [ 0.2183 -3.1668 -0.7574 -2.1793  0.3884  2.103   1.6152  1.5417]
iter   5	rk = [ 0.1636 -3.1976 -0.6366 -2.229   0.2605  2.1442  1.6986  1.5841]
iter   6	rk = [ 0.143  -3.2338 -0.6227 -2.1828  0.2356  2.1546  1.6881  1.6136]
iter   7	rk = [ 0.1322 -3.2353 -0.6116 -2.1749  0.2331  2.198   1.664   1.6005]
iter   8	rk = [ 0.1315 -3.235  -0.6118 -2.175   0.2333  2.1981  1.664   1.6008]
iter   9	rk = [ 0.1315 -3.235  -0.6118 -2.175   0.2333  2.1981  1.664   1.6008]
Solution: [ 0.1315 -3.235  -0.6118 -2.175   0.2333  2.1981  1.664   1.6008]
Info: 0


*(Note: In extreme-scale computations, round-off error can slowly destroy the strict conjugacy of the vectors. Many implementations include a `restart` parameter that periodically flushes the basis history to keep the algorithm stable.)*

## Generalized Minimal Residual Method (GMRES)

What if $\mathbf{A}$ is **not** symmetric? 

If $\mathbf{A}$ is non-symmetric, it cannot be a Hessian, and the beautiful quadratic bowl surface analogy falls apart. We can no longer enforce $\mathbf{A}$-conjugacy.

Instead, we use the **Generalized Minimal Residual (GMRES)** method. GMRES abandons the bowl analogy and directly minimizes the Euclidean norm of the residual $||\mathbf{A}\mathbf{x}-\mathbf{b}||$.

- Beginning with $\vec{x}^0$, construct a basis for $\vec{x}^{approx}-\vec{x}^0$ in $\vec{s}^k$.
- Enforce *orthogonality* with successive $\vec{s}^{k}$
- Since each $\vec{s}^k$ is linearly independent, the method will converge in at most $n$ iterations, but typically converges much sooner.
- A restart may still be necessary to control memory consumption for large systems.

The major difference is that the conjugacy of $\vec{s}^k$ is not possible since $A$ is not SPD. Rather, $\vec{s}^k$ are chosen to be orthogonal which is a more general, but less powerful condition. 

Because it must enforce standard orthogonality against *all* previous search directions, its memory footprint grows with every iteration. Therefore, GMRES almost always requires a `restart` parameter to prevent memory exhaustion on large systems!

In [26]:
from scipy.sparse.linalg import gmres

A = np.array([[4, 1], [1, 3]])
b = np.array([5, 6])
x0 = np.array([0., 0.])

tracker = exe.IterationTracker(x0)

# Execute the native SciPy solver
# Note: callback_type='x' ensures GMRES passes the solution vector instead of the residual
solution, info = gmres(A, b, x0=x0, callback=tracker, callback_type='x')

print("True solution:", np.linalg.solve(A, b))
print("GMRES solution:", solution)

# Visualize the path over the residual surface (since it is not a symmetric positive definite system)
fig = exe.visualize_convergence_2d(A, b, tracker.iterations, surface_type='residual')
fig.show()

True solution: [0.8182 1.7273]
GMRES solution: [0.8182 1.7273]


In [27]:
np.set_printoptions(precision=4)

solution, info = gmres(A_rand, b_rand, x0=x0_rand, atol = 1e-6, rtol = 1e-10, callback = it_counter(), callback_type='legacy')
print("Solution:", solution)
print("Info:", info)

iter   1	rk = 0.4060169853822141
iter   2	rk = 0.30863927568392363
iter   3	rk = 0.156423778024392
iter   4	rk = 0.08571178306778177
iter   5	rk = 0.028900461101078127
iter   6	rk = 0.008223806071236133
iter   7	rk = 4.3278282684119136e-05
iter   8	rk = 2.3418932096143532e-15
Solution: [ 0.1315 -3.235  -0.6118 -2.175   0.2333  2.1981  1.664   1.6008]
Info: 0
